# 01 · Procedencia del dataset y diccionario de datos

**Proyecto:** Predicción del precio de vivienda con aprendizaje automático (UAO · Aprendizaje Automático · 2026-2)  
**Tarjeta de Trello:** `T05` · Procedencia del dataset y diccionario de datos  
**Responsable:** Mauricio José Mozo Terán (Data Engineer)  
**Objetivo:** Verificar la procedencia y relación genealógica de `Housing.csv` (Kaggle) con el dataset original de Windsor (1987) de Anglin & Gençay (JAE / paquete R AER), comprobar por código la equivalencia matemática (precios escalados × 70, registro duplicado en filas 318 y 535, correspondencia de columnas y adición de `furnishingstatus`), construir el diccionario de datos formal con los significados reales y validar la integridad criptográfica (SHA-256), atendiendo el criterio de "Descripción del dataset" de la rúbrica del Entregable 1.

**Contenido**
1. Preparación del entorno
2. Carga y verificación de integridad criptográfica
3. Verificación empírica de procedencia: Windsor 1987 (AER) vs. Housing.csv (Kaggle)
4. Diccionario de datos formal
5. Licencia y marco legal
6. Auditoría preliminar de calidad y conclusiones

## 1. Preparación del entorno

La siguiente celda permite que el notebook se ejecute sin modificaciones tanto en **Google Colab** como en **local**:
- **En Colab:** clona el repositorio privado utilizando el token de GitHub configurado en los secretos (`GITHUB_TOKEN`) e instala las dependencias de `requirements.txt`.
- **En local:** detecta dinámicamente la raíz del repositorio buscando la carpeta `src/` e insertándola en `sys.path`. Esto garantiza que las importaciones de `from src.config import ...` y las rutas relativas funcionen sin importar si el notebook se ejecuta desde la raíz del proyecto o desde la subcarpeta `notebooks/`.

In [1]:
# --- Preparación del entorno (Colab o local) ---
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules  # True si el notebook corre en Google Colab

if EN_COLAB:
    from google.colab import userdata

    repo = "github.com/Juanm0331123/ml-precio-vivienda.git"
    try:
        # Token personal guardado como secreto de Colab (necesario mientras el repo sea privado).
        url = f"https://{userdata.get('GITHUB_TOKEN')}@{repo}"
    except Exception:
        url = f"https://{repo}"  # sirve si el repositorio se vuelve público
    if not Path("ml-precio-vivienda").exists():
        subprocess.run(["git", "clone", "-q", url], check=True)
    os.chdir("ml-precio-vivienda")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Raíz del repositorio = primera carpeta (actual o superior) que contiene src/.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))  # permite: from src.config import ...
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: C:\Users\Windows 11\Desktop\ml-precio-vivienda


**Interpretación del entorno:** Se identificó la raíz del repositorio y se incorporó a `sys.path`. Esto garantiza portabilidad total y previene cualquier error de `ModuleNotFoundError` o `FileNotFoundError` al ejecutar el notebook en entornos Linux, macOS, Windows o Google Colab.

### 1.1 Librerías y configuración compartida

Importamos las librerías científicas requeridas y la configuración centralizada de rutas y reproducibilidad (`src/config.py`).

In [2]:
# --- Librerías y configuración compartida ---
import hashlib
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd

from src.config import (
    DIR_FIGURAS,
    DIR_TABLAS,
    RUTA_DATOS_CRUDOS,
    SHA256_DATOS_CRUDOS,
    RUTA_DATOS_ORIGINAL_AER,
    SHA256_DATOS_ORIGINAL_AER,
    SEMILLA,
)

np.random.seed(SEMILLA)                      # reproducibilidad (semilla global: 5007)
pd.set_option("display.max_columns", 50)     # mostrar todas las columnas
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
DIR_TABLAS.mkdir(parents=True, exist_ok=True)
print("Configuración cargada correctamente. Semilla global:", SEMILLA)

Configuración cargada correctamente. Semilla global:

 5007


**Interpretación:** La semilla del proyecto (5007) y las carpetas de exportación (`reports/figures/`, `reports/tables/`) quedan configuradas para garantizar reproducibilidad exacta en toda la ejecución.

## 2. Carga y verificación de integridad criptográfica

El archivo de trabajo del proyecto es `data/raw/Housing.csv`. Antes de su ingestión, verificamos su integridad mediante su huella criptográfica SHA-256 para comprobar que no ha sufrido alteraciones ni pérdidas de bytes.

In [3]:
# Verificación de integridad SHA-256 de Housing.csv
assert RUTA_DATOS_CRUDOS.exists(), f"No se encontró el archivo en {RUTA_DATOS_CRUDOS}"
huella_calculada = hashlib.sha256(RUTA_DATOS_CRUDOS.read_bytes()).hexdigest()

assert huella_calculada == SHA256_DATOS_CRUDOS, (
    f"Discrepancia en huella: calculada {huella_calculada} != esperada {SHA256_DATOS_CRUDOS}"
)

df_kaggle = pd.read_csv(RUTA_DATOS_CRUDOS)
print("=== VERIFICACIÓN DE INTEGRIDAD CRIPTOGRÁFICA ===")
print(f"Archivo: {RUTA_DATOS_CRUDOS.name}")
print(f"Ruta: {RUTA_DATOS_CRUDOS.relative_to(RAIZ)}")
print(f"SHA-256 calculada: {huella_calculada}")
print(f"SHA-256 esperada:  {SHA256_DATOS_CRUDOS}")
print(f"Estado de integridad: VÁLIDA (coincidencia exacta bit a bit)")
print(f"Dimensiones cargadas: {df_kaggle.shape[0]} filas × {df_kaggle.shape[1]} columnas\n")
df_kaggle.head()

=== VERIFICACIÓN DE INTEGRIDAD CRIPTOGRÁFICA ===
Archivo: Housing.csv
Ruta: data\raw\Housing.csv
SHA-256 calculada: a8ef508351b4d6e87b23a831cf316a3d53c794ed9d969c7b697a865a60a3d65f
SHA-256 esperada:  a8ef508351b4d6e87b23a831cf316a3d53c794ed9d969c7b697a865a60a3d65f
Estado de integridad: VÁLIDA (coincidencia exacta bit a bit)
Dimensiones cargadas: 545 filas × 13 columnas



,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


**Interpretación:** El archivo `Housing.csv` es íntegro y auténtico (su huella SHA-256 en minúsculas es `a8ef508351b4d6e87b23a831cf316a3d53c794ed9d969c7b697a865a60a3d65f`). Cuenta con 545 observaciones y 13 columnas.

## 3. Verificación de procedencia: Windsor 1987 (AER) vs. Housing.csv (Kaggle)

El archivo de Kaggle fue publicado originalmente por Yasser H. bajo el título *"Housing Prices Dataset"*. No obstante, su origen histórico proviene de la investigación econométrica sobre precios hedónicos de viviendas realizada en la ciudad de Windsor, Ontario (Canadá) durante julio, agosto y septiembre de 1987:
- **Publicación:** Anglin, P., y Gençay, R. (1996). *Semiparametric Estimation of a Hedonic Price Function*. Journal of Applied Econometrics, 11(6), 633–648.
- **Distribución:** Paquete `AER` de R (*Applied Econometrics with R*) de Christian Kleiber y Achim Zeileis, distribuido en Rdatasets por Vincent Arel-Bundock.

A continuación, descargamos el archivo original `HousePrices.csv` (si no se encuentra ya guardado en `data/raw/HousePrices_Windsor_1987_AER.csv`), validamos su checksum y contrastamos ambos datasets.

In [4]:
# Descarga y lectura del dataset original de Windsor (paquete AER de R / Rdatasets)
URL_AER = "https://vincentarelbundock.github.io/Rdatasets/csv/AER/HousePrices.csv"

if not RUTA_DATOS_ORIGINAL_AER.exists():
    print(f"Descargando dataset de referencia histórico desde: {URL_AER}")
    urllib.request.urlretrieve(URL_AER, RUTA_DATOS_ORIGINAL_AER)
    print("Descarga finalizada.")

huella_aer = hashlib.sha256(RUTA_DATOS_ORIGINAL_AER.read_bytes()).hexdigest()
assert huella_aer == SHA256_DATOS_ORIGINAL_AER, (
    f"Discrepancia en huella AER: {huella_aer} != {SHA256_DATOS_ORIGINAL_AER}"
)

df_aer = pd.read_csv(RUTA_DATOS_ORIGINAL_AER)

# En Rdatasets la primera columna suele ser el identificador de fila (rownames)
if "rownames" in df_aer.columns:
    df_aer_data = df_aer.drop(columns=["rownames"])
elif "Unnamed: 0" in df_aer.columns:
    df_aer_data = df_aer.drop(columns=["Unnamed: 0"])
else:
    df_aer_data = df_aer.copy()

print("=== DATASET HISTÓRICO ORIGINAL (Windsor 1987 / AER) ===")
print(f"Archivo: {RUTA_DATOS_ORIGINAL_AER.name}")
print(f"SHA-256: {huella_aer}")
print(f"Dimensiones originales: {df_aer_data.shape[0]} filas × {df_aer_data.shape[1]} columnas\n")
df_aer_data.head()

=== DATASET HISTÓRICO ORIGINAL (Windsor 1987 / AER) ===
Archivo: HousePrices_Windsor_1987_AER.csv
SHA-256: 4c7f6df4c6264cf6f57ea6ec5157958c4033efe1f51b2855efb18e297274bfe3
Dimensiones originales: 546 filas × 12 columnas



,price,lotsize,bedrooms,bathrooms,stories,driveway,recreation,fullbase,gasheat,aircon,garage,prefer
0,42000,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500,4000,2,1,1,yes,no,no,no,no,0,no
2,49500,3060,3,1,1,yes,no,no,no,no,0,no
3,60500,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000,6360,2,1,1,yes,no,no,no,no,0,no


**Interpretación:** El dataset histórico original de Windsor posee **546 filas y 12 columnas**, mientras que el dataset de trabajo de Kaggle posee **545 filas y 13 columnas**. A continuación se demuestran con precisión las 4 causas de esta diferencia por medio de código.

### 3.1 Detección del registro duplicado en el archivo original (filas 318 y 535)

Comprobamos si el archivo original de Windsor contiene registros idénticos duplicados que expliquen por qué Kaggle tiene 545 observaciones en vez de 546.

In [5]:
# Identificación de registros duplicados en el archivo original de Windsor
duplicados_aer = df_aer_data.duplicated(keep=False)
filas_duplicadas = df_aer_data[duplicados_aer]

idx_0based = filas_duplicadas.index.tolist()
idx_1based = [i + 1 for i in idx_0based]

print(f"Total de filas duplicadas en el archivo original: {len(filas_duplicadas)}")
print(f"Índices en Python (0-based): {idx_0based}")
print(f"Números de fila en el archivo CSV / R (1-based): {idx_1based}\n")
filas_duplicadas

Total de filas duplicadas en el archivo original: 2
Índices en Python (0-based): [317, 534]
Números de fila en el archivo CSV / R (1-based): [318, 535]



,price,lotsize,bedrooms,bathrooms,stories,driveway,recreation,fullbase,gasheat,aircon,garage,prefer
317,82000,5000,3,1,3,yes,no,no,no,yes,0,no
534,82000,5000,3,1,3,yes,no,no,no,yes,0,no


**Interpretación:** Las filas **318 y 535** del archivo original de Windsor (índices 317 y 534 en Python) contienen exactamente los mismos valores en sus 12 variables: una vivienda vendida en $82,000 CAD, con lote de 5,000 pies², 3 dormitorios, 1 baño, 3 pisos, entrada vehicular (`driveway`), aire acondicionado central (`aircon`) y sin sótano, gas ni sala de juegos. La versión de Kaggle contiene 545 filas porque eliminó esta fila duplicada redundante.

### 3.2 Correspondencia de nombres de columnas y adición de `furnishingstatus`

Comprobamos la correspondencia semántica de las variables y verificamos la existencia de `furnishingstatus` en el dataset original.

In [6]:
# Correspondencia de columnas y verificación de furnishingstatus
mapa_columnas = {
    'price': 'price',
    'area': 'lotsize',
    'bedrooms': 'bedrooms',
    'bathrooms': 'bathrooms',
    'stories': 'stories',
    'mainroad': 'driveway',
    'guestroom': 'recreation',
    'basement': 'fullbase',
    'hotwaterheating': 'gasheat',
    'airconditioning': 'aircon',
    'parking': 'garage',
    'prefarea': 'prefer'
}

cols_kaggle = set(df_kaggle.columns)
cols_original = set(df_aer_data.columns)

print("Columnas en Kaggle que no existen con ese nombre en el original:")
print(cols_kaggle - cols_original)
print("\n¿Existe 'furnishingstatus' en el archivo original?:", 'furnishingstatus' in cols_original)
print("\nFrecuencias de 'furnishingstatus' en Kaggle:")
print(df_kaggle['furnishingstatus'].value_counts())

Columnas en Kaggle que no existen con ese nombre en el original:
{'parking', 'prefarea', 'mainroad', 'area', 'hotwaterheating', 'furnishingstatus', 'airconditioning', 'guestroom', 'basement'}

¿Existe 'furnishingstatus' en el archivo original?: False

Frecuencias de 'furnishingstatus' en Kaggle:
furnishingstatus
semi-furnished    227
unfurnished       178
furnished         140
Name: count, dtype: int64


**Interpretación:** La columna `furnishingstatus` **no existe** en el archivo original de Windsor (1987) ni en el paquete `AER`. Fue añadida sintéticamente en la publicación de Kaggle con tres niveles (`semi-furnished`, `unfurnished`, `furnished`). Las 12 variables restantes corresponden de manera directa a los atributos estructurales y de localización del estudio original.

### 3.3 Verificación del precio: Factor de escala × 70

Verificamos la hipótesis de que los precios en Kaggle corresponden con exactitud al precio original multiplicado por 70 en todas las observaciones.

In [7]:
# Comprobación de que el precio de Kaggle es múltiplo exacto de 70
es_divisible_70 = (df_kaggle['price'] % 70 == 0).all()
print(f"¿Los precios de las 545 filas en Kaggle son múltiplos exactos de 70?: {es_divisible_70}")

precios_kaggle_div70 = (df_kaggle['price'] / 70).astype(int)

print(f"Rango de precios en Windsor original (CAD): [${df_aer_data['price'].min():,} - ${df_aer_data['price'].max():,}]")
print(f"Rango de precios en Kaggle (escalado): [{df_kaggle['price'].min():,} - {df_kaggle['price'].max():,}]")
print(f"Rango de precios en Kaggle divididos entre 70: [${precios_kaggle_div70.min():,} - ${precios_kaggle_div70.max():,}]")

¿Los precios de las 545 filas en Kaggle son múltiplos exactos de 70?: True
Rango de precios en Windsor original (CAD): [$25,000 - $190,000]
Rango de precios en Kaggle (escalado): [1,750,000 - 13,300,000]
Rango de precios en Kaggle divididos entre 70: [$25,000 - $190,000]


**Interpretación:** El 100 % de las 545 filas de Kaggle presenta un precio divisible de forma exacta entre 70. Al dividir por 70, el rango resultante es exactamente [$25,000 – $190,000], idéntico al rango en dólares canadienses (CAD) del estudio original de Windsor de 1987.

### 3.4 Comparación estricta de multiconjuntos (Counter)

Para demostrar con rigor matemático que las 545 filas de Kaggle son exactamente las 545 filas únicas del original, realizamos una comparación de multiconjuntos mediante `collections.Counter`:
1. Dividimos el precio de Kaggle entre 70.
2. Descartamos la columna sintética `furnishingstatus`.
3. Renombramos las columnas de Kaggle a su nombre original.
4. Convertimos cada observación en una tupla y evaluamos si el multiconjunto de Kaggle es un subconjunto exacto del original.

In [8]:
# Comparación rigurosa de multiconjuntos con collections.Counter
df_kaggle_ajustado = df_kaggle.copy()
df_kaggle_ajustado['price'] = (df_kaggle_ajustado['price'] / 70).astype(int)
df_kaggle_ajustado = df_kaggle_ajustado.drop(columns=['furnishingstatus']).rename(columns=mapa_columnas)

# Ordenar las columnas idénticamente al dataset original
orden_columnas = df_aer_data.columns.tolist()
df_kaggle_ajustado = df_kaggle_ajustado[orden_columnas]

# Convertir a tuplas
tuplas_aer = [tuple(x) for x in df_aer_data.to_numpy()]
tuplas_kaggle = [tuple(x) for x in df_kaggle_ajustado.to_numpy()]

conteo_aer = Counter(tuplas_aer)
conteo_kaggle = Counter(tuplas_kaggle)

# Evaluación formal de multiconjunto
diferencia = conteo_aer - conteo_kaggle
es_subconjunto = all(conteo_kaggle[t] <= conteo_aer[t] for t in conteo_kaggle)

print("=== RESULTADO DE LA COMPARACIÓN DE MULTICONJUNTOS ===")
print(f"¿Kaggle es un subconjunto estricto del original de Windsor?: {es_subconjunto}")
print(f"Observaciones en Kaggle: {sum(conteo_kaggle.values())}")
print(f"Observaciones en Windsor original: {sum(conteo_aer.values())}")
print(f"Cardinalidad de la diferencia (Original - Kaggle): {sum(diferencia.values())}")
print(f"Tupla diferencial encontrada:\n{diferencia}")

# Fila esperada del duplicado
fila_dup_esperada = tuple(df_aer_data.iloc[idx_0based[0]])
coincide_duplicado = (list(diferencia.keys())[0] == fila_dup_esperada)
print(f"\n¿La tupla diferencial coincide exactamente con el duplicado de las filas 318 y 535?: {coincide_duplicado}")

=== RESULTADO DE LA COMPARACIÓN DE MULTICONJUNTOS ===
¿Kaggle es un subconjunto estricto del original de Windsor?: True
Observaciones en Kaggle: 545
Observaciones en Windsor original: 546
Cardinalidad de la diferencia (Original - Kaggle): 1
Tupla diferencial encontrada:
Counter({(82000, 5000, 3, 1, 3, 'yes', 'no', 'no', 'no', 'yes', 0, 'no'): 1})

¿La tupla diferencial coincide exactamente con el duplicado de las filas 318 y 535?: True


**Interpretación:** La prueba matemática es categórica:
1. Las 545 observaciones de `Housing.csv` coinciden exactamente con las del dataset original de Windsor (1987) tras dividir el precio entre 70 y alinear las variables.
2. La diferencia entre ambos multiconjuntos consta de exactamente **1 elemento**: la observación redundante de las filas 318 y 535 (`price` = 82000, `lotsize` = 5000, `bedrooms` = 3, `bathrooms` = 1, `stories` = 3, `driveway` = yes, `recreation` = no, `fullbase` = no, `gasheat` = no, `aircon` = yes, `garage` = 0, `prefer` = no).
3. Queda demostrada con absoluta certeza la procedencia del dataset de trabajo y su equivalencia con el estudio de Windsor (1987).

## 4. Diccionario de datos formal

Construimos el diccionario de datos formal para el dataset de trabajo (`data/raw/Housing.csv`) con las 7 columnas solicitadas: `variable en Kaggle | nombre original | descripción real | tipo | unidad o niveles | rango | rol`, reflejando el significado real de cada variable en el contexto residencial de Windsor y exportándolo a `reports/tables/diccionario_datos.csv`.

In [9]:
# Construcción programática del diccionario de datos formal
filas_diccionario = [
    {
        'variable en Kaggle': 'price',
        'nombre original': 'price',
        'descripción real': 'Precio final de venta del inmueble (escalado en Kaggle como precio original × 70)',
        'tipo': 'Numérica continua',
        'unidad o niveles': 'Moneda (unidades monetarias escaladas)',
        'rango': f"[{df_kaggle['price'].min():,} - {df_kaggle['price'].max():,}]",
        'rol': 'Objetivo (Target)'
    },
    {
        'variable en Kaggle': 'area',
        'nombre original': 'lotsize',
        'descripción real': 'Tamaño total del lote o terreno de la propiedad',
        'tipo': 'Numérica continua',
        'unidad o niveles': 'Pies cuadrados (sq ft)',
        'rango': f"[{df_kaggle['area'].min():,} - {df_kaggle['area'].max():,}]",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'bedrooms',
        'nombre original': 'bedrooms',
        'descripción real': 'Número de dormitorios o habitaciones en la vivienda',
        'tipo': 'Numérica discreta',
        'unidad o niveles': 'Entero (cuartos)',
        'rango': f"[{df_kaggle['bedrooms'].min()} - {df_kaggle['bedrooms'].max()}]",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'bathrooms',
        'nombre original': 'bathrooms',
        'descripción real': 'Número de cuartos de baño completos',
        'tipo': 'Numérica discreta',
        'unidad o niveles': 'Entero (baños)',
        'rango': f"[{df_kaggle['bathrooms'].min()} - {df_kaggle['bathrooms'].max()}]",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'stories',
        'nombre original': 'stories',
        'descripción real': 'Número de pisos o plantas habitables sobre rasante (excluye el sótano)',
        'tipo': 'Numérica discreta',
        'unidad o niveles': 'Entero (niveles)',
        'rango': f"[{df_kaggle['stories'].min()} - {df_kaggle['stories'].max()}]",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'mainroad',
        'nombre original': 'driveway',
        'descripción real': 'Presencia de entrada vehicular pavimentada hacia la propiedad (driveway)',
        'tipo': 'Categórica binaria',
        'unidad o niveles': 'yes, no',
        'rango': "{'no', 'yes'}",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'guestroom',
        'nombre original': 'recreation',
        'descripción real': 'Presencia de sala de recreación o cuarto de juegos (recreation room)',
        'tipo': 'Categórica binaria',
        'unidad o niveles': 'yes, no',
        'rango': "{'no', 'yes'}",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'basement',
        'nombre original': 'fullbase',
        'descripción real': 'Presencia de sótano completo terminado y acondicionado (full finished basement)',
        'tipo': 'Categórica binaria',
        'unidad o niveles': 'yes, no',
        'rango': "{'no', 'yes'}",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'hotwaterheating',
        'nombre original': 'gasheat',
        'descripción real': 'Uso de gas para calentar el agua (gas water heating)',
        'tipo': 'Categórica binaria',
        'unidad o niveles': 'yes, no',
        'rango': "{'no', 'yes'}",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'airconditioning',
        'nombre original': 'aircon',
        'descripción real': 'Presencia de sistema de aire acondicionado central',
        'tipo': 'Categórica binaria',
        'unidad o niveles': 'yes, no',
        'rango': "{'no', 'yes'}",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'parking',
        'nombre original': 'garage',
        'descripción real': 'Número de plazas de estacionamiento disponibles en el garaje',
        'tipo': 'Numérica discreta',
        'unidad o niveles': 'Entero (plazas)',
        'rango': f"[{df_kaggle['parking'].min()} - {df_kaggle['parking'].max()}]",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'prefarea',
        'nombre original': 'prefer',
        'descripción real': 'Ubicación en barrio o vecindario preferencial de la ciudad',
        'tipo': 'Categórica binaria',
        'unidad o niveles': 'yes, no',
        'rango': "{'no', 'yes'}",
        'rol': 'Predictora (Feature)'
    },
    {
        'variable en Kaggle': 'furnishingstatus',
        'nombre original': '(Añadida en Kaggle)',
        'descripción real': 'Estado de amueblamiento de la vivienda (añadida en Kaggle, sin documentación en el original)',
        'tipo': 'Categórica politómica',
        'unidad o niveles': 'furnished, semi-furnished, unfurnished',
        'rango': "{'furnished', 'semi-furnished', 'unfurnished'}",
        'rol': 'Predictora (Feature)'
    }
]

df_diccionario = pd.DataFrame(filas_diccionario)

# Exportar tabla CSV
ruta_dict_csv = DIR_TABLAS / "diccionario_datos.csv"
df_diccionario.to_csv(ruta_dict_csv, index=False, encoding="utf-8")
print(f"Diccionario de datos exportado a: {ruta_dict_csv.relative_to(RAIZ)}")
df_diccionario

Diccionario de datos exportado a: reports\tables\diccionario_datos.csv


,variable en Kaggle,nombre original,descripción real,tipo,unidad o niveles,rango,rol
0,price,price,Precio final de venta del inmueble (escalado e...,Numérica continua,Moneda (unidades monetarias escaladas),"[1,750,000 - 13,300,000]",Objetivo (Target)
1,area,lotsize,Tamaño total del lote o terreno de la propiedad,Numérica continua,Pies cuadrados (sq ft),"[1,650 - 16,200]",Predictora (Feature)
2,bedrooms,bedrooms,Número de dormitorios o habitaciones en la viv...,Numérica discreta,Entero (cuartos),[1 - 6],Predictora (Feature)
3,bathrooms,bathrooms,Número de cuartos de baño completos,Numérica discreta,Entero (baños),[1 - 4],Predictora (Feature)
4,stories,stories,Número de pisos o plantas habitables sobre ras...,Numérica discreta,Entero (niveles),[1 - 4],Predictora (Feature)
5,mainroad,driveway,Presencia de entrada vehicular pavimentada hac...,Categórica binaria,"yes, no","{'no', 'yes'}",Predictora (Feature)
6,guestroom,recreation,Presencia de sala de recreación o cuarto de ju...,Categórica binaria,"yes, no","{'no', 'yes'}",Predictora (Feature)
7,basement,fullbase,Presencia de sótano completo terminado y acond...,Categórica binaria,"yes, no","{'no', 'yes'}",Predictora (Feature)
8,hotwaterheating,gasheat,Uso de gas para calentar el agua (gas water he...,Categórica binaria,"yes, no","{'no', 'yes'}",Predictora (Feature)
9,airconditioning,aircon,Presencia de sistema de aire acondicionado cen...,Categórica binaria,"yes, no","{'no', 'yes'}",Predictora (Feature)


**Interpretación:** El diccionario de datos aclara el significado real de cada campo según la literatura econométrica de Windsor: `mainroad` es la entrada vehicular privada (`driveway`), `guestroom` es una sala de esparcimiento (`recreation room`), `basement` es un sótano completamente terminado (`fullbase`), `area` es la superficie del terreno en pies² (`lotsize`), y `furnishingstatus` es una columna sintética agregada en Kaggle. La variable `price` es el target objetivo y las 12 restantes son predictoras.

## 5. Licencia y marco legal

Se documenta formalmente la situación legal de ambos archivos, distinguiendo con claridad entre licencias copyleft GPL y Dominio Público.

In [10]:
# Marco legal y resumen de licencias
licencias_info = [
    {
        'Componente': 'Datos originales de Windsor (1987)',
        'Fuente': 'Journal of Applied Econometrics Archive / Paquete R AER',
        'Licencia': 'GNU General Public License (GPL-2 | GPL-3)',
        'Tipo Legal': 'Copyleft (Software Libre / Open Data)',
        'Atribución': 'Obligatoria: Citar formalmente a Anglin & Gençay (1996)',
        'Permisos': 'Uso académico, investigación y análisis reproducible'
    },
    {
        'Componente': 'Dataset de trabajo (Housing.csv)',
        'Fuente': 'Kaggle (Yasser H. - Housing Prices Dataset)',
        'Licencia': 'CC0: Public Domain (Creative Commons Zero 1.0)',
        'Tipo Legal': 'Dominio Público',
        'Atribución': 'No requerida legalmente (mantenida por rigor científico)',
        'Permisos': 'Libre uso comercial, educativo y analítico sin restricciones'
    }
]

df_licencias = pd.DataFrame(licencias_info)
ruta_lic_csv = DIR_TABLAS / "marco_legal.csv"
df_licencias.to_csv(ruta_lic_csv, index=False, encoding="utf-8")
print(f"Tabla de marco legal exportada a: {ruta_lic_csv.relative_to(RAIZ)}")
df_licencias

Tabla de marco legal exportada a: reports\tables\marco_legal.csv


,Componente,Fuente,Licencia,Tipo Legal,Atribución,Permisos
0,Datos originales de Windsor (1987),Journal of Applied Econometrics Archive / Paqu...,GNU General Public License (GPL-2 | GPL-3),Copyleft (Software Libre / Open Data),Obligatoria: Citar formalmente a Anglin & Genç...,"Uso académico, investigación y análisis reprod..."
1,Dataset de trabajo (Housing.csv),Kaggle (Yasser H. - Housing Prices Dataset),CC0: Public Domain (Creative Commons Zero 1.0),Dominio Público,No requerida legalmente (mantenida por rigor c...,"Libre uso comercial, educativo y analítico sin..."


**Interpretación:** GPL y Dominio Público no son equivalentes. El dataset original de Windsor está protegido bajo GPL-2/3, lo que exige licenciamiento abierto para derivados y reconocimiento de autoría a Anglin & Gençay (1996). La versión en Kaggle cuenta con dedicación CC0 (Dominio Público).

## 6. Auditoría preliminar de calidad y conclusiones

Realizamos un perfilado de completitud inicial sobre el archivo de trabajo `Housing.csv`.

In [11]:
# Auditoría preliminar de completitud de Housing.csv
resumen_calidad = pd.DataFrame({
    'Variable': df_kaggle.columns,
    'Tipo': [str(df_kaggle[c].dtype) for c in df_kaggle.columns],
    'Valores Nulos': df_kaggle.isnull().sum().values,
    '% Nulos': (df_kaggle.isnull().sum().values / len(df_kaggle)) * 100,
    'Completitud %': 100.0 - ((df_kaggle.isnull().sum().values / len(df_kaggle)) * 100),
    'Valores Únicos': [df_kaggle[c].nunique() for c in df_kaggle.columns]
})

duplicados_kaggle = df_kaggle.duplicated().sum()
print("=== RESUMEN INICIAL DE CALIDAD: Housing.csv ===")
print(f"Total observaciones: {len(df_kaggle)}")
print(f"Filas duplicadas: {duplicados_kaggle}")
print(f"Total valores nulos: {df_kaggle.isnull().sum().sum()}\n")
resumen_calidad

=== RESUMEN INICIAL DE CALIDAD: Housing.csv ===
Total observaciones: 545
Filas duplicadas: 0
Total valores nulos: 0



,Variable,Tipo,Valores Nulos,% Nulos,Completitud %,Valores Únicos
0,price,int64,0,0.0,100.0,219
1,area,int64,0,0.0,100.0,284
2,bedrooms,int64,0,0.0,100.0,6
3,bathrooms,int64,0,0.0,100.0,4
4,stories,int64,0,0.0,100.0,4
5,mainroad,str,0,0.0,100.0,2
6,guestroom,str,0,0.0,100.0,2
7,basement,str,0,0.0,100.0,2
8,hotwaterheating,str,0,0.0,100.0,2
9,airconditioning,str,0,0.0,100.0,2


**Interpretación y Conclusiones de la Tarjeta T05:**
1. **Procedencia validada:** Se comprobó empírica y matemáticamente con multiconjuntos (`collections.Counter`) que `Housing.csv` proviene del dataset histórico de Windsor (1987) publicado por Anglin & Gençay (1996), con sus precios multiplicados exactamente por 70 y habiendo depurado el duplicado que presentaba el original en las filas 318 y 535.
2. **Integridad:** El archivo de trabajo `Housing.csv` cuenta con una huella SHA-256 verificada (`a8ef508351b4d6e87b23a831cf316a3d53c794ed9d969c7b697a865a60a3d65f`) y presenta 100 % de completitud en sus 545 registros.
3. **Diccionario de datos formal:** Se construyó el diccionario con las 7 columnas requeridas y las definiciones semánticas reales, exportado a `reports/tables/diccionario_datos.csv`.
4. **Marco legal:** Se documentó el licenciamiento dual (GPL-2/3 para el original y CC0 para Kaggle).
5. **Paso a T06:** El análisis exhaustivo de calidad, correlaciones bivariadas, detección de outliers, evaluación de colinealidad y partición train/test sellada se desarrollará en la tarjeta **T06**.